# Task 2: Visualizing fire scars through false color

https://github.com/veronica-moran/eds220-hwk4.git

# About the data

In this task you will use two datasets. The first is a simplified collection of bands (red, green, blue, near-infrared and shortwave infrared) from the Landsat Collection 2 Level-2 atmospherically corrected surface reflectance data, collected by the Landsat 8 satellite.

The data was retrieved from the [Microsoft Planetary Computer data catalog](https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2) and clipped to an area surrounding the fire perimeters. This data should be used for visualization and educational purposes only and can be accessed as landsat8-2025-02-23-palisades-eaton.nc in the course shared data [Google Drive](https://drive.google.com/drive/u/1/folders/1USqhiMLyN8GE05B8WJmHabviJGnmAsLP).

The second dataset will be fire perimeters for the Eaton and Palisades fires. There are several datasets with this information online. You will need to independently select one from a reputable source (it may be more than one file).

[AVIRIS Data Portal](https://popo.jpl.nasa.gov/mmgis-aviris/?mission=LA_Fires)

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import xarray as xr
from shapely.geometry import box
pd.set_option('display.max_columns', None)

In [2]:
# Read fire perimeters
fp = os.path.join('data', 'fire_perimeters.geojson')
perimeters = gpd.read_file(fp)

# Read Landsat imagery
fp = os.path.join('data', 'landsat8-2025-02-23-palisades-eaton.nc')
bands = xr.open_dataset(fp)

sh: 1: getfattr: not found


# Fire Perimeter Data Exploration

In [4]:
perimeters.head()

,name,acres,created,Shape__Are,Shape__Len,source,dateTime,_,geometry
0,AUTO,55.7,1736839224000,3.305966e+05,3633.829649,FIRIS,2025-01-14 07:20:24+00:00,"{'idx': 34, 'end_time': '1736839224000'}","MULTIPOLYGON (((-119.21282 34.23610, -119.2128..."
1,PALISADES,23706.6,1736686649000,1.401755e+08,129483.127792,INTEL FLIGHT DATA,2025-01-12 12:57:29+00:00,"{'idx': 33, 'end_time': '1736686649000'}","MULTIPOLYGON (((-118.60006 34.08394, -118.6000..."
2,Eaton,14005.4,1736686830000,8.306198e+07,101455.890112,INTEL FLIGHT DATA,2025-01-12 13:00:30+00:00,"{'idx': 32, 'end_time': '1736686830000'}","MULTIPOLYGON (((-118.13628 34.17676, -118.1366..."
3,Palisades,23713.4,1736707013000,1.402157e+08,129567.976023,INTEL FLIGHT DATA,2025-01-12 18:36:53+00:00,"{'idx': 31, 'end_time': '1736707013000'}","MULTIPOLYGON (((-118.60006 34.08394, -118.6000..."
4,Eaton,13991.6,1736622854000,8.298039e+07,101053.743729,INTEL FLIGHT DATA,2025-01-11 19:14:14+00:00,"{'idx': 30, 'end_time': '1736622854000'}","MULTIPOLYGON (((-118.13628 34.17676, -118.1366..."


In [5]:
perimeters.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   name        34 non-null     object             
 1   acres       34 non-null     object             
 2   created     34 non-null     int64              
 3   Shape__Are  34 non-null     float64            
 4   Shape__Len  34 non-null     float64            
 5   source      34 non-null     object             
 6   dateTime    34 non-null     datetime64[ns, UTC]
 7   _           34 non-null     object             
 8   geometry    34 non-null     geometry           
dtypes: datetime64[ns, UTC](1), float64(2), geometry(1), int64(1), object(4)
memory usage: 2.5+ KB


In [6]:
# Check if the CRS is geographic
print(perimeters.crs.is_geographic)

# Check if the CRS is projected
print(perimeters.crs.is_projected)

perimeters.crs

True
False


<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [7]:
perimeters['name'].unique()

array(['AUTO', 'PALISADES', 'Eaton', 'Palisades', 'EATON', 'ARCHER',
       'KENNETH', 'HURST', 'LIDIA', 'SUNSET', 'UNK'], dtype=object)

The fire perimeter dataset contains 34 rows and 9 columns, with no missing values. The data include fire names, acreage, perimeter area and length, data sources, observation dates, and fire perimeter geometries. There are nine fire names which includes different capitalizations for Palisades and Eaton fires and one unknown fire name. Acres dtype is an object, instead of numeric. The CRS is WGS 84 (EPSG:4326), a geographic reference system.

# NetCDF Data Exploration